In [60]:
# https://github.com/jeandeducla/ML-Time-Series/blob/master/SVM-Accelerometer.ipynb
import pandas as pd
import numpy as np
import datetime as dt

In [61]:
import scipy.stats as st
from scipy.fftpack import fft, fftfreq 
from scipy.signal import argrelextrema
import operator

In [169]:
df=pd.read_csv('data_ref_until_2020-02-13.csv')

In [170]:
df.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue
0,2020-01-25 00:01:49.251503+01:00,20.64,55.57,121.0,496.0,521.0,694.0,652.0,591.0,135.0
1,2020-01-25 00:02:09.350082+01:00,20.64,55.53,122.0,495.0,521.0,694.0,652.0,591.0,135.0
2,2020-01-25 00:02:29.319505+01:00,20.62,55.53,121.0,495.0,522.0,694.0,651.0,591.0,135.0
3,2020-01-25 00:02:49.399379+01:00,20.62,55.57,121.0,495.0,522.0,694.0,651.0,591.0,135.0
4,2020-01-25 00:03:09.158989+01:00,20.62,55.59,121.0,495.0,522.0,694.0,652.0,591.0,135.0


In [171]:
# need to convert time string to time format for easy processing
df['recordtime'] = df['timestamp']
df.recordtime =df.recordtime.apply(lambda x: pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S"))
df.recordtime =df.recordtime.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [172]:
# sort values from small time to large time
# how much signal is difference from the prevous time to current time is important
df.sort_values(by='recordtime', inplace=True, ascending=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['timestamp'] = df['timestamp'].dt.floor(freq='1H')

In [173]:
# always check the latest data
df.head()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue,recordtime
0,2020-01-25 00:00:00+01:00,20.64,55.57,121.0,496.0,521.0,694.0,652.0,591.0,135.0,2020-01-25 00:01:49
1,2020-01-25 00:00:00+01:00,20.64,55.53,122.0,495.0,521.0,694.0,652.0,591.0,135.0,2020-01-25 00:02:09
2,2020-01-25 00:00:00+01:00,20.62,55.53,121.0,495.0,522.0,694.0,651.0,591.0,135.0,2020-01-25 00:02:29
3,2020-01-25 00:00:00+01:00,20.62,55.57,121.0,495.0,522.0,694.0,651.0,591.0,135.0,2020-01-25 00:02:49
4,2020-01-25 00:00:00+01:00,20.62,55.59,121.0,495.0,522.0,694.0,652.0,591.0,135.0,2020-01-25 00:03:09


In [174]:
df.tail()

,timestamp,temperature,humidity,CO2CosIRValue,CO2MG811Value,MOX1,MOX2,MOX3,MOX4,COValue,recordtime
75506,2020-02-13 11:00:00+01:00,20.62,57.75,62.0,518.0,524.0,698.0,642.0,591.0,157.0,2020-02-13 11:56:53
75507,2020-02-13 11:00:00+01:00,20.62,57.75,62.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:57:14
75508,2020-02-13 11:00:00+01:00,20.62,57.75,62.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:57:33
75509,2020-02-13 11:00:00+01:00,20.62,57.74,63.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:57:54
75510,2020-02-13 11:00:00+01:00,20.62,57.74,63.0,518.0,524.0,698.0,641.0,591.0,157.0,2020-02-13 11:58:14


In [175]:
df.shape

(75511, 11)

In [176]:
# look the count for each time group
group_count = pd.DataFrame(df.groupby('timestamp').count()['humidity'])
group_count.reset_index(inplace=True)
group_count.rename(columns={"humidity": "countv"}, inplace=True)

In [177]:
# some of the time group has low number of data missing
# Fillup them
group_count.countv.unique()

array([175, 180, 125, 157, 164, 171, 162,  86, 178, 146, 148,  47,  94,
       145,  98, 179, 181])

In [178]:
# these time group has low count - 
# lot of data is missing for these. So they need to be removed
time_low_count = group_count[group_count.countv<150]
time_low_count

,timestamp,countv
22,2020-01-25 22:00:00+01:00,125
31,2020-01-26 07:00:00+01:00,86
35,2020-01-26 11:00:00+01:00,146
36,2020-01-26 12:00:00+01:00,148
206,2020-02-02 14:00:00+01:00,47
207,2020-02-04 12:00:00+01:00,94
245,2020-02-06 02:00:00+01:00,145
246,2020-02-06 03:00:00+01:00,98


In [179]:
# np.unique([(list(df.recordtime)[i] - list(df.recordtime)[i-1]).seconds for i in range(1, len(df.recordtime))])

In [180]:
df.columns

Index(['timestamp', 'temperature', 'humidity', 'CO2CosIRValue',
       'CO2MG811Value', 'MOX1', 'MOX2', 'MOX3', 'MOX4', 'COValue',
       'recordtime'],
      dtype='object')

In [181]:
# investigate how these data is missing
processed_df = df[['temperature', 'humidity', 'CO2CosIRValue','CO2MG811Value', 'MOX1', 'MOX4', 'COValue']].copy()
processed_df = processed_df.rolling(3).mean()
processed_df['timestamp'] = df['timestamp']
processed_df['recordtime'] = df['recordtime']

In [182]:
# ndf = pd.DataFrame(columns=df.columns)
# for time in df.recordtime:
#     current_value = list(df[df.recordtime==time].values[0])
#     ndf.loc[len(ndf)] = current_value

#     next_time = time + dt.timedelta(seconds=22)
#     temp = df[(df.recordtime>time) & (df.recordtime<=next_time)]

#     if len(temp) < 1:
#         current_value[-1] = next_time
#         ndf.loc[len(ndf)] = current_value

#         next_time = next_time + dt.timedelta(seconds=22)
#         temp = df[(df.recordtime>time) & (df.recordtime<=next_time)]

#         if len(temp) < 1:
#             current_value[-1] = next_time
#             ndf.loc[len(ndf)] = current_value

In [183]:
processed_df=processed_df[~processed_df.timestamp.isin(time_low_count.timestamp)]

In [184]:
processed_df.shape

(74622, 9)

In [185]:
group_count = group_count[~group_count.timestamp.isin(time_low_count.timestamp)]

In [186]:
group_count.shape

(415, 2)

In [187]:
def stat_area_features(x, Te=1.0):

    mean_ts = np.mean(x, axis=1).reshape(-1,1) # mean
    max_ts = np.amax(x, axis=1).reshape(-1,1) # max
    min_ts = np.amin(x, axis=1).reshape(-1,1) # min
    std_ts = np.std(x, axis=1).reshape(-1,1) # std
    skew_ts = st.skew(x, axis=1).reshape(-1,1) # skew
    kurtosis_ts = st.kurtosis(x, axis=1).reshape(-1,1) # kurtosis 
    iqr_ts = st.iqr(x, axis=1).reshape(-1,1) # interquartile rante
    mad_ts = np.median(np.sort(abs(x - np.median(x, axis=1).reshape(-1,1)),
                               axis=1), axis=1).reshape(-1,1) # median absolute deviation
    area_ts = np.trapz(x, axis=1, dx=Te).reshape(-1,1) # area under curve
    sq_area_ts = np.trapz(x ** 2, axis=1, dx=Te).reshape(-1,1) # area under curve ** 2

    return np.concatenate((mean_ts,max_ts,min_ts,std_ts,skew_ts,kurtosis_ts,
                           iqr_ts,mad_ts,area_ts,sq_area_ts), axis=1)

In [188]:
def frequency_domain_features(x, Te=1.0):

    # As the DFT coefficients and their corresponding frequencies are symetrical arrays
    # with respect to the middle of the array we need to know if the number of readings 
    # in x is even or odd to then split the arrays...
    if x.shape[1]%2 == 0:
        N = int(x.shape[1]/2)
    else:
        N = int(x.shape[1]/2) - 1
    xf = np.repeat(fftfreq(x.shape[1],d=Te)[:N].reshape(1,-1), x.shape[0], axis=0) # frequencies
    dft = np.abs(fft(x, axis=1))[:,:N] # DFT coefficients   
    
    # statistical and area features
    dft_features = stat_area_features(dft, Te=1.0)
    # weighted mean frequency
    dft_weighted_mean_f = np.average(xf, axis=1, weights=dft).reshape(-1,1)
    # 5 first DFT coefficients 
    dft_first_coef = dft[:,:5]    
    # 5 first local maxima of DFT coefficients and their corresponding frequencies
    dft_max_coef = np.zeros((x.shape[0],5))
    dft_max_coef_f = np.zeros((x.shape[0],5))
    for row in range(x.shape[0]):
        # finds all local maximas indexes
        extrema_ind = argrelextrema(dft[row,:], np.greater, axis=0) 
        # makes a list of tuples (DFT_i, f_i) of all the local maxima
        # and keeps the 5 biggest...
        extrema_row = sorted([(dft[row,:][j],xf[row,j]) for j in extrema_ind[0]],
                             key=operator.itemgetter(0), reverse=True)[:5] 
        for i, ext in enumerate(extrema_row):
            dft_max_coef[row,i] = ext[0]
            dft_max_coef_f[row,i] = ext[1]    
    
    return np.concatenate((dft_features,dft_weighted_mean_f,dft_first_coef,
                           dft_max_coef,dft_max_coef_f), axis=1)


In [189]:
def convert_to_list(data, column_name):
    Unique_time = data.timestamp.unique()
    X = []
    for time in Unique_time:
        values = data[data.timestamp==time][column_name].tolist()
        if len(values)<181:
            resample_len=181-len(values)
            new_data = [np.mean(values)] * resample_len
            values.extend(new_data)
        X.append(values)
    return np.array(X)

In [190]:
def make_feature(data, column_name, Te=1.0):
    # convert data into list of lists
    x=convert_to_list(data, column_name)

    # extract statistical feature
    x_stat=stat_area_features(x)
    x_stat=pd.DataFrame(x_stat)
    x_stat.columns=[column_name+'_stat_'+ str(i) for i in range(x_stat.shape[1])]

    # extract statistical feature for the signal difference
    xt_jerk = stat_area_features((x[:, 1:]-x[:, :-1])/Te, Te=Te)
    xt_jerk = pd.DataFrame(xt_jerk)
    xt_jerk.columns=[column_name + '_xt_jerk_' + str(i) for i in range(xt_jerk.shape[1])]
    x_final=pd.concat([x_stat, xt_jerk], axis=1)

    
    # extract frequency_domain_features
    x_domain=frequency_domain_features(x)
    x_domain=pd.DataFrame(x_domain)
    x_domain.columns=[column_name +'_tm_dom_'+ str(i) for i in range(x_domain.shape[1])]
    x_final = pd.concat([x_final, x_domain], axis=1)

    # extract frequency_domain_features for the signal difference
    xf_jerk = frequency_domain_features((x[:, 1:]-x[:, :-1])/Te, Te=1/Te)
    xf_jerk = pd.DataFrame(xf_jerk)
    xf_jerk.columns=[column_name + '_xf_jerk_' + str(i) for i in range(xf_jerk.shape[1])]
    x_final = pd.concat([x_final, xf_jerk], axis=1)

    return x_final

In [191]:
# test only for humidity column
humidity=make_feature(processed_df, column_name='humidity')

In [192]:
humidity.head()

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_xf_jerk_16,humidity_xf_jerk_17,humidity_xf_jerk_18,humidity_xf_jerk_19,humidity_xf_jerk_20,humidity_xf_jerk_21,humidity_xf_jerk_22,humidity_xf_jerk_23,humidity_xf_jerk_24,humidity_xf_jerk_25
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,51.499444,57.190000,48.266667,2.979673,0.516279,-1.068425,5.086667,2.453333,9267.174722,478706.144717,...,8.171612,7.472853,6.314694,5.784242,5.150218,0.016667,0.055556,0.033333,0.005556,0.105556
2,49.259944,50.066667,48.030000,0.465519,-0.131263,-0.530683,0.720000,0.386667,8867.059972,436843.258874,...,3.016065,2.925024,2.379540,2.180107,2.164740,0.055556,0.038889,0.111111,0.077778,0.005556
3,48.548481,49.260000,47.800000,0.329641,-0.067398,-0.118018,0.323333,0.170000,8738.735907,424272.474877,...,5.761894,3.053225,2.087130,1.486311,1.388314,0.055556,0.027778,0.138889,0.233333,0.127778
4,48.633685,49.293333,48.030000,0.310936,0.040159,-0.541608,0.410000,0.220000,8754.281843,425781.017895,...,3.123322,3.116170,3.026981,1.669182,1.640669,0.044444,0.027778,0.055556,0.011111,0.072222


In [193]:
humidity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 72 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   humidity_stat_0      414 non-null    float64
 1   humidity_stat_1      414 non-null    float64
 2   humidity_stat_2      414 non-null    float64
 3   humidity_stat_3      414 non-null    float64
 4   humidity_stat_4      414 non-null    float64
 5   humidity_stat_5      414 non-null    float64
 6   humidity_stat_6      414 non-null    float64
 7   humidity_stat_7      414 non-null    float64
 8   humidity_stat_8      414 non-null    float64
 9   humidity_stat_9      414 non-null    float64
 10  humidity_xt_jerk_0   414 non-null    float64
 11  humidity_xt_jerk_1   414 non-null    float64
 12  humidity_xt_jerk_2   414 non-null    float64
 13  humidity_xt_jerk_3   414 non-null    float64
 14  humidity_xt_jerk_4   414 non-null    float64
 15  humidity_xt_jerk_5   414 non-null    flo

In [194]:
humidity.describe()

,humidity_stat_0,humidity_stat_1,humidity_stat_2,humidity_stat_3,humidity_stat_4,humidity_stat_5,humidity_stat_6,humidity_stat_7,humidity_stat_8,humidity_stat_9,...,humidity_xf_jerk_16,humidity_xf_jerk_17,humidity_xf_jerk_18,humidity_xf_jerk_19,humidity_xf_jerk_20,humidity_xf_jerk_21,humidity_xf_jerk_22,humidity_xf_jerk_23,humidity_xf_jerk_24,humidity_xf_jerk_25
count,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,...,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000
mean,56.676569,57.382174,55.974984,0.387902,0.004361,-0.470272,0.594802,0.263506,10201.775402,579251.178057,...,2.125763,1.650290,1.399208,1.235678,1.100456,0.030830,0.048206,0.062972,0.082008,0.100602
std,2.351606,2.449609,2.425017,0.407239,0.586593,0.925227,0.744594,0.324537,423.290989,46865.162667,...,1.683541,1.173627,1.018361,0.939593,0.842786,0.022082,0.034532,0.041655,0.052297,0.058400
min,47.262815,47.553333,46.270000,0.058328,-2.212339,-1.776089,0.086667,0.040000,8507.191407,402071.379143,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,55.312926,55.872500,54.865000,0.214384,-0.345838,-1.019664,0.290000,0.134167,9956.317713,550724.559202,...,1.350818,1.066568,0.897464,0.788983,0.687733,0.011111,0.022222,0.033333,0.044444,0.061111
50%,57.040250,57.685000,56.390000,0.288839,0.007423,-0.704539,0.400000,0.180000,10267.238458,585656.598718,...,1.759451,1.370280,1.174019,1.035207,0.921501,0.027778,0.038889,0.061111,0.072222,0.094444
75%,58.267902,58.909167,57.626667,0.387866,0.384320,-0.159215,0.582500,0.262500,10488.179130,611133.926307,...,2.278970,1.868383,1.549411,1.343965,1.174353,0.044444,0.061111,0.077778,0.122222,0.138889
max,63.264667,69.260000,62.390000,3.421664,2.181374,5.509277,6.823333,3.190000,11387.602333,720548.380135,...,15.459732,9.939001,8.259913,8.005218,6.871095,0.166667,0.216667,0.211111,0.488889,0.488889


In [195]:
# Get for all columns
features_df = group_count.copy()
for column_name in processed_df.columns:
    if column_name not in ("timestamp", 'recordtime'):
        ndf=make_feature(processed_df, column_name=column_name)
        features_df=pd.concat([features_df, ndf], axis=1)
    print(column_name)

temperature
humidity
CO2CosIRValue
CO2MG811Value
MOX1
MOX4
COValue
timestamp
recordtime


In [196]:
features_df.shape

(423, 506)

In [197]:
features_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 423 entries, 0 to 246
Columns: 506 entries, timestamp to COValue_xf_jerk_25
dtypes: datetime64[ns, pytz.FixedOffset(60)](1), float64(505)
memory usage: 1.6 MB


In [198]:
features_df.head()

,timestamp,countv,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,...,COValue_xf_jerk_16,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25
0,2020-01-25 00:00:00+01:00,175.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2020-01-25 01:00:00+01:00,180.0,19.570722,20.190000,19.133333,0.277370,0.602720,-0.505364,0.396667,0.173333,...,14.028797,13.400307,13.277110,12.649683,12.201266,0.044444,0.033333,0.066667,0.155556,0.016667
2,2020-01-25 02:00:00+01:00,180.0,18.800944,19.160000,18.616667,0.132610,0.554188,-0.313950,0.166667,0.080000,...,15.018556,14.313917,13.039316,12.323973,11.894470,0.055556,0.183333,0.038889,0.133333,0.238889
3,2020-01-25 03:00:00+01:00,180.0,18.471815,18.686667,18.260000,0.112353,0.043721,-0.931267,0.180000,0.076667,...,8.727736,6.014104,5.611913,5.579166,5.519712,0.155556,0.216667,0.050000,0.133333,0.027778
4,2020-01-25 04:00:00+01:00,180.0,18.037907,18.350000,17.830000,0.092318,0.203885,0.637195,0.120000,0.056667,...,9.927526,8.262032,7.379868,6.666471,6.504218,0.050000,0.094444,0.127778,0.172222,0.038889


In [199]:
features_df.describe()

,countv,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,temperature_stat_8,...,COValue_xf_jerk_16,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25
count,415.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,414.000000,...,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000,415.000000
mean,179.812048,20.052315,20.288221,19.781892,0.140441,-0.196271,-0.578793,0.210918,0.096815,3609.421199,...,9.887963,8.268062,7.472399,6.860299,6.342479,0.085448,0.095515,0.107068,0.123440,0.134096
std,1.724602,1.356910,1.392141,1.324994,0.078714,0.554081,0.835548,0.148318,0.065051,244.252611,...,10.236523,7.891437,6.803247,5.849676,4.376189,0.052627,0.057778,0.072304,0.083721,0.090460
min,157.000000,15.086000,15.200000,14.950000,0.035012,-1.724871,-1.733431,0.046667,0.020000,2715.508000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,180.000000,19.987042,20.210000,19.690000,0.096598,-0.557817,-1.067792,0.127500,0.060000,3597.656021,...,7.115171,6.165404,5.619750,5.175386,4.895205,0.044444,0.050000,0.055556,0.066667,0.072222
50%,180.000000,20.195231,20.423333,19.935000,0.129524,-0.212557,-0.773640,0.183333,0.083333,3635.220949,...,8.299624,7.094967,6.300587,5.834208,5.463151,0.061111,0.094444,0.100000,0.111111,0.127778
75%,180.000000,20.641625,20.860000,20.367500,0.159777,0.128314,-0.288863,0.243333,0.116667,3715.525813,...,9.938245,8.149282,7.431313,6.781267,6.289190,0.133333,0.133333,0.144444,0.161111,0.166667
max,181.000000,23.078907,23.470000,22.780000,0.784604,2.176464,7.129384,1.630000,0.673333,4154.332787,...,180.588975,146.567179,126.148995,105.303128,69.583898,0.238889,0.488889,0.488889,0.488889,0.488889


In [200]:
features_stat_df = pd.DataFrame(features_df.describe())

In [201]:
processed_features_df = features_df.copy()
processed_features_df['timestamp'] = features_df['timestamp']
processed_features_df.dropna(inplace=True)
processed_features_df.timestamp = processed_features_df.timestamp.apply(lambda x: pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S"))
processed_features_df.timestamp = processed_features_df.timestamp.apply(lambda x: dt.datetime.strptime(x, "%Y-%m-%d %H:%M:%S"))

In [202]:
processed_features_df.head()

,timestamp,countv,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,...,COValue_xf_jerk_16,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25
1,2020-01-25 01:00:00,180.0,19.570722,20.190000,19.133333,0.277370,0.602720,-0.505364,0.396667,0.173333,...,14.028797,13.400307,13.277110,12.649683,12.201266,0.044444,0.033333,0.066667,0.155556,0.016667
2,2020-01-25 02:00:00,180.0,18.800944,19.160000,18.616667,0.132610,0.554188,-0.313950,0.166667,0.080000,...,15.018556,14.313917,13.039316,12.323973,11.894470,0.055556,0.183333,0.038889,0.133333,0.238889
3,2020-01-25 03:00:00,180.0,18.471815,18.686667,18.260000,0.112353,0.043721,-0.931267,0.180000,0.076667,...,8.727736,6.014104,5.611913,5.579166,5.519712,0.155556,0.216667,0.050000,0.133333,0.027778
4,2020-01-25 04:00:00,180.0,18.037907,18.350000,17.830000,0.092318,0.203885,0.637195,0.120000,0.056667,...,9.927526,8.262032,7.379868,6.666471,6.504218,0.050000,0.094444,0.127778,0.172222,0.038889
5,2020-01-25 05:00:00,180.0,17.862722,18.006667,17.670000,0.090997,-0.376686,-0.850565,0.136667,0.070000,...,9.252283,9.021639,9.014348,7.926246,5.624210,0.111111,0.061111,0.072222,0.155556,0.127778


In [203]:
processed_features_df['timestamp']

1     2020-01-25 01:00:00
2     2020-01-25 02:00:00
3     2020-01-25 03:00:00
4     2020-01-25 04:00:00
5     2020-01-25 05:00:00
              ...        
410   2020-02-12 23:00:00
411   2020-02-13 00:00:00
412   2020-02-13 01:00:00
413   2020-02-13 02:00:00
414   2020-02-13 03:00:00
Name: timestamp, Length: 406, dtype: datetime64[ns]

In [204]:
processed_features_df.describe()

,countv,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,temperature_stat_8,...,COValue_xf_jerk_16,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25
count,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,...,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000,406.000000
mean,179.832512,20.075562,20.311379,19.805714,0.140385,-0.195253,-0.577875,0.210952,0.096858,3613.603962,...,9.963670,8.324755,7.521205,6.908327,6.386912,0.084962,0.095361,0.107157,0.124275,0.135003
std,1.710253,1.316702,1.349176,1.287451,0.078156,0.555462,0.840238,0.148814,0.065087,237.014546,...,10.329287,7.961595,6.862827,5.899004,4.406824,0.051402,0.057891,0.072542,0.083943,0.090815
min,157.000000,15.086000,15.200000,14.950000,0.035135,-1.724871,-1.733431,0.046667,0.020000,2715.508000,...,4.571800,3.921342,3.449362,3.402505,3.316873,0.005556,0.005556,0.005556,0.005556,0.005556
25%,180.000000,19.991069,20.217500,19.700000,0.096725,-0.557817,-1.068469,0.127500,0.060000,3598.434285,...,7.134756,6.196040,5.643992,5.192714,4.929717,0.044444,0.050000,0.055556,0.066667,0.072222
50%,180.000000,20.195231,20.423333,19.935000,0.129656,-0.212557,-0.766252,0.183333,0.083333,3635.220949,...,8.308434,7.100496,6.316274,5.837838,5.484555,0.061111,0.091667,0.100000,0.111111,0.127778
75%,180.000000,20.641625,20.860000,20.370000,0.159777,0.138678,-0.288863,0.243333,0.116667,3715.525813,...,9.972697,8.163438,7.459022,6.809160,6.309016,0.133333,0.133333,0.144444,0.161111,0.172222
max,181.000000,23.078907,23.470000,22.780000,0.784604,2.176464,7.129384,1.630000,0.673333,4154.332787,...,180.588975,146.567179,126.148995,105.303128,69.583898,0.238889,0.488889,0.488889,0.488889,0.488889


In [205]:

# Adding a new column named 'total_count' with float data type and all values set to 0
processed_features_df = processed_features_df.assign(total_count=0.0)

In [206]:
processed_features_df

,timestamp,countv,temperature_stat_0,temperature_stat_1,temperature_stat_2,temperature_stat_3,temperature_stat_4,temperature_stat_5,temperature_stat_6,temperature_stat_7,...,COValue_xf_jerk_17,COValue_xf_jerk_18,COValue_xf_jerk_19,COValue_xf_jerk_20,COValue_xf_jerk_21,COValue_xf_jerk_22,COValue_xf_jerk_23,COValue_xf_jerk_24,COValue_xf_jerk_25,total_count
1,2020-01-25 01:00:00,180.0,19.570722,20.190000,19.133333,0.277370,0.602720,-0.505364,0.396667,0.173333,...,13.400307,13.277110,12.649683,12.201266,0.044444,0.033333,0.066667,0.155556,0.016667,0.0
2,2020-01-25 02:00:00,180.0,18.800944,19.160000,18.616667,0.132610,0.554188,-0.313950,0.166667,0.080000,...,14.313917,13.039316,12.323973,11.894470,0.055556,0.183333,0.038889,0.133333,0.238889,0.0
3,2020-01-25 03:00:00,180.0,18.471815,18.686667,18.260000,0.112353,0.043721,-0.931267,0.180000,0.076667,...,6.014104,5.611913,5.579166,5.519712,0.155556,0.216667,0.050000,0.133333,0.027778,0.0
4,2020-01-25 04:00:00,180.0,18.037907,18.350000,17.830000,0.092318,0.203885,0.637195,0.120000,0.056667,...,8.262032,7.379868,6.666471,6.504218,0.050000,0.094444,0.127778,0.172222,0.038889,0.0
5,2020-01-25 05:00:00,180.0,17.862722,18.006667,17.670000,0.090997,-0.376686,-0.850565,0.136667,0.070000,...,9.021639,9.014348,7.926246,5.624210,0.111111,0.061111,0.072222,0.155556,0.127778,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410,2020-02-12 23:00:00,180.0,20.273537,20.583333,20.076667,0.144346,0.998453,-0.063589,0.136667,0.066667,...,11.331225,9.277357,8.943533,8.640065,0.044444,0.172222,0.083333,0.016667,0.150000,0.0
411,2020-02-13 00:00:00,180.0,20.344130,20.610000,20.116667,0.175572,0.141872,-1.599176,0.360000,0.153333,...,9.822270,8.915631,8.817341,8.402870,0.050000,0.155556,0.144444,0.061111,0.022222,0.0
412,2020-02-13 01:00:00,180.0,20.394019,20.570000,20.090000,0.170591,-0.561510,-1.352172,0.343333,0.083333,...,6.706358,6.704544,4.995000,3.988314,0.055556,0.005556,0.016667,0.077778,0.033333,0.0
413,2020-02-13 02:00:00,180.0,20.337111,20.590000,20.080000,0.124831,0.113164,-0.555568,0.183333,0.070000,...,8.053292,7.791950,7.447059,6.596604,0.027778,0.066667,0.050000,0.111111,0.416667,0.0


In [207]:
keep_list=['CO2CosIRValue_stat_3',
 'CO2CosIRValue_stat_9',
 'CO2CosIRValue_xt_jerk_1',
 'CO2CosIRValue_xt_jerk_5',
 'CO2CosIRValue_xt_jerk_9',
 'CO2CosIRValue_tm_dom_2',
 'CO2CosIRValue_tm_dom_16',
 'CO2CosIRValue_xf_jerk_2',
 'CO2CosIRValue_xf_jerk_15',
 'COValue_stat_9',
 'COValue_xt_jerk_1',
 'total_count']

In [42]:
keep_list.append('timestamp')

In [208]:
final_features_df = processed_features_df[keep_list].copy()
final_features_df.head()

,CO2CosIRValue_stat_3,CO2CosIRValue_stat_9,CO2CosIRValue_xt_jerk_1,CO2CosIRValue_xt_jerk_5,CO2CosIRValue_xt_jerk_9,CO2CosIRValue_tm_dom_2,CO2CosIRValue_tm_dom_16,CO2CosIRValue_xf_jerk_2,CO2CosIRValue_xf_jerk_15,COValue_stat_9,COValue_xt_jerk_1,total_count
1,17.832044,1.718756e+06,25.146296,153.069952,374.001442,2.422827,170.906919,12.514679,48.505221,2.652015e+06,9.288889,0.0
2,2.609765,7.812039e+05,3.174074,32.893747,18.370706,0.473141,34.736728,0.666369,6.025589,1.943643e+06,8.827778,0.0
3,0.925873,6.878950e+05,1.000000,1.405384,16.591221,0.284929,49.395343,0.359963,4.044760,1.580783e+06,0.379630,0.0
4,1.080819,6.919446e+05,1.000000,2.201301,20.716682,0.267050,67.050694,0.275039,3.614868,1.476063e+06,1.548148,0.0
5,1.005143,6.849507e+05,0.666667,2.666603,12.048210,0.077875,69.689594,0.230430,5.324749,1.383061e+06,1.655556,0.0


In [209]:
final_features_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 406 entries, 1 to 414
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CO2CosIRValue_stat_3      406 non-null    float64
 1   CO2CosIRValue_stat_9      406 non-null    float64
 2   CO2CosIRValue_xt_jerk_1   406 non-null    float64
 3   CO2CosIRValue_xt_jerk_5   406 non-null    float64
 4   CO2CosIRValue_xt_jerk_9   406 non-null    float64
 5   CO2CosIRValue_tm_dom_2    406 non-null    float64
 6   CO2CosIRValue_tm_dom_16   406 non-null    float64
 7   CO2CosIRValue_xf_jerk_2   406 non-null    float64
 8   CO2CosIRValue_xf_jerk_15  406 non-null    float64
 9   COValue_stat_9            406 non-null    float64
 10  COValue_xt_jerk_1         406 non-null    float64
 11  total_count               406 non-null    float64
dtypes: float64(12)
memory usage: 41.2 KB


In [156]:
del final_features_df['timestamp']

KeyError: 'timestamp'

In [210]:
final_features_df.to_csv('Mean_data_no_candidate_v2.csv')

In [211]:
final_features_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 406 entries, 1 to 414
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CO2CosIRValue_stat_3      406 non-null    float64
 1   CO2CosIRValue_stat_9      406 non-null    float64
 2   CO2CosIRValue_xt_jerk_1   406 non-null    float64
 3   CO2CosIRValue_xt_jerk_5   406 non-null    float64
 4   CO2CosIRValue_xt_jerk_9   406 non-null    float64
 5   CO2CosIRValue_tm_dom_2    406 non-null    float64
 6   CO2CosIRValue_tm_dom_16   406 non-null    float64
 7   CO2CosIRValue_xf_jerk_2   406 non-null    float64
 8   CO2CosIRValue_xf_jerk_15  406 non-null    float64
 9   COValue_stat_9            406 non-null    float64
 10  COValue_xt_jerk_1         406 non-null    float64
 11  total_count               406 non-null    float64
dtypes: float64(12)
memory usage: 41.2 KB


In [212]:
drop_features = ['Living room', 'Bedroom', 'Bathroom', 'Kitchen', 'Unnamed: 0']

In [213]:
df=pd.read_csv('Mean_data_processed_v2.csv')

In [214]:
df = df.drop(drop_features, axis=1)

In [215]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CO2CosIRValue_stat_3      1531 non-null   float64
 1   CO2CosIRValue_stat_9      1531 non-null   float64
 2   CO2CosIRValue_xt_jerk_1   1531 non-null   float64
 3   CO2CosIRValue_xt_jerk_5   1531 non-null   float64
 4   CO2CosIRValue_xt_jerk_9   1531 non-null   float64
 5   CO2CosIRValue_tm_dom_2    1531 non-null   float64
 6   CO2CosIRValue_tm_dom_16   1531 non-null   float64
 7   CO2CosIRValue_xf_jerk_2   1531 non-null   float64
 8   CO2CosIRValue_xf_jerk_15  1531 non-null   float64
 9   COValue_stat_9            1531 non-null   float64
 10  COValue_xt_jerk_1         1531 non-null   float64
 11  total_count               1531 non-null   float64
dtypes: float64(12)
memory usage: 143.7 KB


In [53]:
df.rename(columns={'total_count.1': 'total_count'}, inplace=True)

In [216]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CO2CosIRValue_stat_3      1531 non-null   float64
 1   CO2CosIRValue_stat_9      1531 non-null   float64
 2   CO2CosIRValue_xt_jerk_1   1531 non-null   float64
 3   CO2CosIRValue_xt_jerk_5   1531 non-null   float64
 4   CO2CosIRValue_xt_jerk_9   1531 non-null   float64
 5   CO2CosIRValue_tm_dom_2    1531 non-null   float64
 6   CO2CosIRValue_tm_dom_16   1531 non-null   float64
 7   CO2CosIRValue_xf_jerk_2   1531 non-null   float64
 8   CO2CosIRValue_xf_jerk_15  1531 non-null   float64
 9   COValue_stat_9            1531 non-null   float64
 10  COValue_xt_jerk_1         1531 non-null   float64
 11  total_count               1531 non-null   float64
dtypes: float64(12)
memory usage: 143.7 KB


In [217]:
frame=[df,final_features_df]

In [218]:
final_data=pd.concat(frame)

In [219]:
final_data

,CO2CosIRValue_stat_3,CO2CosIRValue_stat_9,CO2CosIRValue_xt_jerk_1,CO2CosIRValue_xt_jerk_5,CO2CosIRValue_xt_jerk_9,CO2CosIRValue_tm_dom_2,CO2CosIRValue_tm_dom_16,CO2CosIRValue_xf_jerk_2,CO2CosIRValue_xf_jerk_15,COValue_stat_9,COValue_xt_jerk_1,total_count
0,4.277250,1.297384e+06,0.942857,1.039894,9.709796,0.462801,55.628117,0.054267,1.048591,1.641997e+06,0.285714,12.0
1,4.661196,1.215978e+06,2.064286,9.193603,16.844923,0.207858,83.595212,0.523647,9.918481,2.234549e+06,0.857143,2.0
2,2.928801,1.328604e+06,0.571429,5.519146,16.599365,1.668825,109.318905,0.573488,10.099731,2.098801e+06,0.411111,8.0
3,0.839037,1.363174e+06,1.165204,6.035382,6.908720,0.186527,47.409887,0.155241,4.268468,2.108054e+06,0.428571,9.0
4,1.316341,2.001046e+06,0.571429,5.530090,10.319739,0.729999,86.283489,0.267901,11.038523,2.591721e+06,0.285714,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...
410,0.779258,3.367352e+05,1.911111,8.488250,14.103951,0.095617,40.077413,0.192167,2.802264,3.954769e+06,0.333333,0.0
411,0.985778,3.310420e+05,0.666667,0.807783,14.441360,0.383580,61.915949,0.110469,7.428919,4.022870e+06,4.488889,0.0
412,1.150586,3.327721e+05,1.000000,2.368824,18.244583,0.341689,79.406058,0.323018,1.397248,3.983293e+06,0.666667,0.0
413,0.818716,3.431804e+05,0.666667,0.152468,14.272867,0.039042,51.512676,0.157784,1.026341,4.064812e+06,0.666667,0.0


In [220]:
final_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1937 entries, 0 to 414
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CO2CosIRValue_stat_3      1937 non-null   float64
 1   CO2CosIRValue_stat_9      1937 non-null   float64
 2   CO2CosIRValue_xt_jerk_1   1937 non-null   float64
 3   CO2CosIRValue_xt_jerk_5   1937 non-null   float64
 4   CO2CosIRValue_xt_jerk_9   1937 non-null   float64
 5   CO2CosIRValue_tm_dom_2    1937 non-null   float64
 6   CO2CosIRValue_tm_dom_16   1937 non-null   float64
 7   CO2CosIRValue_xf_jerk_2   1937 non-null   float64
 8   CO2CosIRValue_xf_jerk_15  1937 non-null   float64
 9   COValue_stat_9            1937 non-null   float64
 10  COValue_xt_jerk_1         1937 non-null   float64
 11  total_count               1937 non-null   float64
dtypes: float64(12)
memory usage: 196.7 KB


In [221]:
final_data.to_csv('Merge_data_mean_v5.csv')